# Paper 2 — Multimodal Radiogenomic Prediction of Phenoconversion
### Prodromal-to-PD conversion analysis pipeline (PPMI)

Clean, de-duplicated version of the analysis code for this paper only.
(Separated from the original motor-progression-prediction notebook, which lives in a different repository.)


## Part A — Build the analytic cohort from the raw PPMI Curated Data Cut

Run this once. It saves `PPMI_Prodromal_Conversion_Analytic_Cohort.csv` to Google Drive; after that, Part B can just read the saved file directly.

In [ ]:
# ============================================================
# PAPER 2: PRODROMAL PD CONVERSION COHORT
# Build final time-to-conversion analytic dataset
# Restricted to the official Prodromal cohort (COHORT == 4)
# excludes Healthy Controls and other non-PD baseline groups.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

INPUT_FILE = Path(
    "/content/drive/MyDrive/Colab Notebooks/Paper2_Radiogenomics/data/PPMI_Curated_Data_Cut_Public_20260810.xlsx"
)
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/PPMI_Genetics/Paper2_Conversion"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "PPMI_Prodromal_Conversion_Analytic_Cohort.csv"

print("Reading PPMI data...")
df = pd.read_excel(INPUT_FILE, sheet_name="20260810")
print(f"Full dataset: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

genetic_vars = ["GP2_proxy_PGS", "META5_proxy_PGS"]
datscan_vars = ["MIA_PUTAMEN_L", "MIA_PUTAMEN_R", "MIA_CAUDATE_L", "MIA_CAUDATE_R",
                "MIA_STRIATUM_L", "MIA_STRIATUM_R"]
clinical_vars = ["age", "SEX", "updrs3_score", "BMI", "NP1DPRS", "MSEADLG"]
required_vars = (["PATNO", "EVENT_ID", "COHORT", "PRIMDIAG", "visit_date"]
                  + genetic_vars + datscan_vars + clinical_vars)

missing_vars = [v for v in required_vars if v not in df.columns]
if missing_vars:
    raise ValueError(f"The following required variables are missing:\n{missing_vars}")

df["visit_date"] = pd.to_datetime(df["visit_date"], errors="coerce")

baseline = df[(df["EVENT_ID"] == "BL") & (df["PRIMDIAG"] != 1) & (df["COHORT"] == 4)].copy()
print(f"\nBaseline Prodromal (COHORT=4), non-PD participants: {baseline['PATNO'].nunique():,}")

baseline = baseline.sort_values(["PATNO", "visit_date"]).drop_duplicates("PATNO", keep="first")

predictor_vars = genetic_vars + datscan_vars + clinical_vars
baseline_complete = baseline.dropna(subset=predictor_vars).copy()
print(f"Complete baseline cases for all predictors: {baseline_complete['PATNO'].nunique():,}")

candidate_patnos = set(baseline_complete["PATNO"].astype(str))
df["_PATNO_STR"] = df["PATNO"].astype(str)
followup = df[df["_PATNO_STR"].isin(candidate_patnos)].copy()
baseline_dates = baseline_complete[["PATNO", "visit_date"]].rename(columns={"visit_date": "baseline_date"})
followup = followup.merge(baseline_dates, on="PATNO", how="inner")
followup_after_baseline = followup[followup["visit_date"] > followup["baseline_date"]].copy()

conversion_visits = followup_after_baseline[followup_after_baseline["PRIMDIAG"] == 1].copy()
first_conversion = (conversion_visits.sort_values(["PATNO", "visit_date"])
                     .groupby("PATNO", as_index=False).first()[["PATNO", "visit_date"]]
                     .rename(columns={"visit_date": "conversion_date"}))
print(f"Participants with a later PD diagnosis: {first_conversion['PATNO'].nunique():,}")

last_followup = (followup_after_baseline.dropna(subset=["visit_date"])
                  .groupby("PATNO", as_index=False)["visit_date"].max()
                  .rename(columns={"visit_date": "last_followup_date"}))

analytic = baseline_complete.merge(first_conversion, on="PATNO", how="left")
analytic = analytic.merge(last_followup, on="PATNO", how="left")
analytic["event"] = (analytic["conversion_date"].notna()).astype(int)
analytic["end_date"] = np.where(analytic["event"] == 1, analytic["conversion_date"], analytic["last_followup_date"])
analytic["end_date"] = pd.to_datetime(analytic["end_date"], errors="coerce")
analytic["follow_up_days"] = (analytic["end_date"] - analytic["visit_date"]).dt.days
analytic["follow_up_years"] = analytic["follow_up_days"] / 365.25

final_columns = (["PATNO", "COHORT", "visit_date", "conversion_date", "last_followup_date", "end_date",
                   "event", "follow_up_days", "follow_up_years"]
                  + clinical_vars + datscan_vars + genetic_vars)
analytic_final = analytic[final_columns].copy()
analytic_final = analytic_final.rename(columns={"visit_date": "baseline_date"})
analytic_final = analytic_final[analytic_final["follow_up_days"].notna() & (analytic_final["follow_up_days"] >= 0)].copy()

analytic_final.to_csv(OUTPUT_FILE, index=False)

print("\n" + "=" * 65)
print("FINAL COHORT VERIFICATION (Prodromal-only, COHORT=4)")
print("=" * 65)
print(f"Total analytic participants: {len(analytic_final):,}")
print(f"Converters: {analytic_final['event'].sum():,}")
print(f"Non-converters: {(analytic_final['event'] == 0).sum():,}")
print(f"Conversion proportion: {analytic_final['event'].mean() * 100:.2f}%")
print("\nSaved to:", OUTPUT_FILE)
display(analytic_final.head())


## Part B — Setup and load the analytic cohort

In [ ]:
!pip install lifelines statsmodels -q

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

np.random.seed(42)
pd.set_option('display.width', 150)
print("Setup complete.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import files

from pathlib import Path

COHORT_FILE = Path(
    "/content/drive/MyDrive/Colab Notebooks/PPMI_Genetics/Paper2_Conversion/"
    "PPMI_Prodromal_Conversion_Analytic_Cohort.csv"
)

if not COHORT_FILE.exists():
    raise FileNotFoundError(
        f"File not found at: {COHORT_FILE}\n"
        "Run Part A first to build the analytic cohort."
    )

full_complete = pd.read_csv(COHORT_FILE)
full_complete = full_complete.rename(columns={'follow_up_years': 'duration'})
if full_complete['SEX'].dtype == object:
    full_complete['SEX'] = full_complete['SEX'].astype('category').cat.codes

print(f"Cohort shape: {full_complete.shape}")
print(f"Converters: {(full_complete['event']==1).sum()}")
print(f"Non-converters: {(full_complete['event']==0).sum()}")
print(f"Conversion rate: {100*full_complete['event'].mean():.2f}%")
full_complete.head()


## Part C — Add monogenic LRRK2/GBA carrier status (secondary genetic representation)

Upload the PPMI "Genetic Data - Consensus APOE Genotype and Pathogenic Variants" file when prompted.

In [ ]:
print("Upload the mutation carrier file (iu_genetic_consensus_...csv)")
uploaded_mut = files.upload()
mut_filename = list(uploaded_mut.keys())[0]

mut = pd.read_csv(mut_filename)
mut['LRRK2_carrier'] = mut['LRRK2'].apply(lambda x: np.nan if pd.isna(x) else (0 if x == '0' else 1))
mut['GBA_carrier'] = mut['GBA'].apply(lambda x: np.nan if pd.isna(x) else (0 if x == '0' else 1))

full_complete = full_complete.merge(mut[['PATNO', 'LRRK2_carrier', 'GBA_carrier']], on='PATNO', how='left')
print(f"Matched with mutation data: {full_complete['LRRK2_carrier'].notna().sum()} of {len(full_complete)}")

before = len(full_complete)
full_complete = full_complete.dropna(subset=['LRRK2_carrier', 'GBA_carrier']).copy()
print(f"Cohort after requiring mutation data too: {len(full_complete)} (dropped {before - len(full_complete)})")
print(f"Converters: {(full_complete['event']==1).sum()}, Non-converters: {(full_complete['event']==0).sum()}")


## Part D — Define predictor sets and candidate models

In [ ]:
dat_cols = ['MIA_PUTAMEN_L', 'MIA_PUTAMEN_R', 'MIA_CAUDATE_L', 'MIA_CAUDATE_R',
            'MIA_STRIATUM_L', 'MIA_STRIATUM_R']
gen_cols = ['GP2_proxy_PGS', 'META5_proxy_PGS']              # PRS only (primary analysis)
gen_cols_mut = ['LRRK2_carrier', 'GBA_carrier']               # mutation carrier status only
gen_cols_all = gen_cols + gen_cols_mut                        # PRS + mutation status combined
clin_cols = ['age', 'SEX', 'updrs3_score', 'BMI', 'NP1DPRS', 'MSEADLG']

feature_cols = dat_cols + gen_cols_all + clin_cols
missing = [c for c in feature_cols if c not in full_complete.columns]
if missing:
    print("WARNING - missing columns:", missing)
else:
    print("All feature columns present.")
    print(f"DaTSCAN={len(dat_cols)}, Genetics(PRS)={len(gen_cols)}, "
          f"Genetics(mutations)={len(gen_cols_mut)}, Clinical={len(clin_cols)}")


In [ ]:
MODELS = {
    'M1_Clinical':               clin_cols,
    'M2_Genetics_PRS':           gen_cols,
    'M2b_Genetics_Mutations':    gen_cols_mut,
    'M2c_Genetics_All':          gen_cols_all,
    'M3_DaTSCAN':                dat_cols,
    'M4_Genetics_Clinical':      gen_cols + clin_cols,
    'M4b_Genetics_All_Clinical': gen_cols_all + clin_cols,
    'M5_DaTSCAN_Clinical':       dat_cols + clin_cols,
    'M6_DaTSCAN_Genetics_PRS':   dat_cols + gen_cols,
    'M6b_DaTSCAN_Genetics_All':  dat_cols + gen_cols_all,
    'M7_Full_Fusion_PRS':        dat_cols + gen_cols + clin_cols,
    'M7b_Full_Fusion_All':       dat_cols + gen_cols_all + clin_cols,
}
for name, cols in MODELS.items():
    print(f"{name}: {len(cols)} features")


## Part E — Cox proportional hazards model: hazard ratios (full feature set)

In [ ]:
scaler = StandardScaler()
X_all = full_complete[dat_cols + gen_cols_all + clin_cols].copy()
X_all_scaled = pd.DataFrame(scaler.fit_transform(X_all), columns=X_all.columns, index=X_all.index)

cph_full = pd.concat([full_complete[['duration', 'event']].reset_index(drop=True),
                       X_all_scaled.reset_index(drop=True)], axis=1)

cph = CoxPHFitter(penalizer=0.1)
cph.fit(cph_full, duration_col='duration', event_col='event')
cph.print_summary()

hr_table = cph.summary[['coef', 'exp(coef)', 'p']].sort_values('p')
hr_table.to_csv('Paper2_HazardRatios_FullModel.csv')
files.download('Paper2_HazardRatios_FullModel.csv')


## Part F — Cross-validated discrimination (C-index) for every model

In [ ]:
def cv_concordance(cols, n_splits=5, n_repeats=20, penalizer=0.1):
    data = full_complete[['duration', 'event'] + cols].copy()
    X = StandardScaler().fit_transform(data[cols])
    data_scaled = pd.concat([data[['duration', 'event']].reset_index(drop=True),
                              pd.DataFrame(X, columns=cols).reset_index(drop=True)], axis=1)
    scores = []
    for repeat in range(n_repeats):
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=repeat)
        for train_idx, test_idx in kf.split(data_scaled):
            train, test = data_scaled.iloc[train_idx], data_scaled.iloc[test_idx]
            if train['event'].sum() < 5:
                continue
            try:
                m = CoxPHFitter(penalizer=penalizer)
                m.fit(train, duration_col='duration', event_col='event')
                pred = m.predict_partial_hazard(test)
                c = concordance_index(test['duration'], -pred, test['event'])
                scores.append(c)
            except Exception:
                continue
    return scores

results = {}
for name, cols in MODELS.items():
    scores = cv_concordance(cols)
    results[name] = scores
    print(f"{name}: C-index = {np.mean(scores):.4f} (SD={np.std(scores):.4f}, n_folds={len(scores)})")


## Part G — Formal pairwise statistical comparisons

In [ ]:
def paired_comparison(scores_a, scores_b, label):
    n = min(len(scores_a), len(scores_b))
    a, b = np.array(scores_a[:n]), np.array(scores_b[:n])
    diff = a - b
    t_stat, p_val = stats.ttest_1samp(diff, 0)
    print(f"{label}: mean_diff={diff.mean():.4f}, p={p_val:.4f}")
    return p_val

print("=== DaTSCAN+Clinical vs Full Fusion (PRS only) ===")
paired_comparison(results['M7_Full_Fusion_PRS'], results['M5_DaTSCAN_Clinical'], 'Fusion(PRS) vs DaTSCAN+Clinical')
print()
print("=== DaTSCAN+Clinical vs Full Fusion (PRS + mutations) ===")
paired_comparison(results['M7b_Full_Fusion_All'], results['M5_DaTSCAN_Clinical'], 'Fusion(All genetics) vs DaTSCAN+Clinical')
print()
print("=== Full Fusion (PRS only) vs Full Fusion (PRS + mutations) ===")
paired_comparison(results['M7b_Full_Fusion_All'], results['M7_Full_Fusion_PRS'], 'Fusion(All) vs Fusion(PRS only)')
print()
print("=== Genetics PRS-only vs Genetics with mutations ===")
paired_comparison(results['M2c_Genetics_All'], results['M2_Genetics_PRS'], 'Genetics(All) vs Genetics(PRS only)')
print()
print("=== Clinical alone vs Genetics(All)+Clinical -- does genetics help WITHOUT imaging? ===")
paired_comparison(results['M4b_Genetics_All_Clinical'], results['M1_Clinical'], 'Genetics(All)+Clinical vs Clinical alone')
print()
print("=== DaTSCAN-only vs Full Fusion (PRS) ===")
paired_comparison(results['M7_Full_Fusion_PRS'], results['M3_DaTSCAN'], 'Fusion(PRS) vs DaTSCAN-only')
print()
print("=== Genetics-only vs Clinical-only ===")
paired_comparison(results['M2_Genetics_PRS'], results['M1_Clinical'], 'Genetics(PRS) vs Clinical')

summary_rows = [{'model': k, 'mean_cindex': np.mean(v), 'sd_cindex': np.std(v), 'n_folds': len(v)} for k, v in results.items()]
summary_df = pd.DataFrame(summary_rows).sort_values('mean_cindex', ascending=False)
summary_df.to_csv('Paper2_ModelComparison_Cindex.csv', index=False)
files.download('Paper2_ModelComparison_Cindex.csv')
summary_df


## Part H — Permutation-based predictor importance (primary PRS-only model, matches Table 4)

In [ ]:
def permutation_importance_survival(cols, n_repeats=10):
    data = full_complete[['duration', 'event'] + cols].copy()
    X = StandardScaler().fit_transform(data[cols])
    data_scaled = pd.concat([data[['duration', 'event']].reset_index(drop=True),
                              pd.DataFrame(X, columns=cols).reset_index(drop=True)], axis=1)

    m = CoxPHFitter(penalizer=0.1)
    m.fit(data_scaled, duration_col='duration', event_col='event')
    pred = m.predict_partial_hazard(data_scaled)
    baseline_c = concordance_index(data_scaled['duration'], -pred, data_scaled['event'])

    importances = {}
    for col in cols:
        drops = []
        for _ in range(n_repeats):
            shuffled = data_scaled.copy()
            shuffled[col] = np.random.permutation(shuffled[col].values)
            pred_s = m.predict_partial_hazard(shuffled)
            c_s = concordance_index(shuffled['duration'], -pred_s, shuffled['event'])
            drops.append(baseline_c - c_s)
        importances[col] = np.mean(drops)
    return baseline_c, importances

baseline_c, importances = permutation_importance_survival(dat_cols + gen_cols + clin_cols)
print(f"Baseline C-index: {baseline_c:.4f}")
imp_df = pd.DataFrame(importances.items(), columns=['feature', 'importance_drop']).sort_values('importance_drop', ascending=False)
imp_df.to_csv('Paper2_PermutationImportance.csv', index=False)
imp_df


## Part I — Sensitivity check: dropping the striatum variables (collinear with putamen/caudate)

In [ ]:
dat_cols_no_striatum = ['MIA_PUTAMEN_L', 'MIA_PUTAMEN_R', 'MIA_CAUDATE_L', 'MIA_CAUDATE_R']
scores_sensitivity = cv_concordance(dat_cols_no_striatum + gen_cols + clin_cols)
print(f"Full model WITHOUT striatum: C-index = {np.mean(scores_sensitivity):.4f} (SD={np.std(scores_sensitivity):.4f})")
print(f"Full model WITH striatum (M7_Full_Fusion_PRS): C-index = {np.mean(results['M7_Full_Fusion_PRS']):.4f} (SD={np.std(results['M7_Full_Fusion_PRS']):.4f})")


## Part J — Figures

In [ ]:
# ===== Figure 1: C-index comparison across the 7 primary models (M1-M7, PRS-only) =====
primary_models = ['M1_Clinical', 'M2_Genetics_PRS', 'M3_DaTSCAN', 'M4_Genetics_Clinical',
                   'M5_DaTSCAN_Clinical', 'M6_DaTSCAN_Genetics_PRS', 'M7_Full_Fusion_PRS']
means = [np.mean(results[m]) for m in primary_models]
sds = [np.std(results[m]) for m in primary_models]
colors = ['green' if 'Full' in m else 'steelblue' for m in primary_models]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(primary_models, means, yerr=sds, capsize=4, color=colors, edgecolor='black')
ax.set_ylabel('Concordance index (C-index)')
ax.set_title('Cross-validated discrimination by model')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('figure1_cindex_comparison.png', dpi=300)
plt.show()
files.download('figure1_cindex_comparison.png')

# ===== Figure 2: Permutation importance (primary PRS-only model) =====
imp_sorted = sorted(importances.items(), key=lambda x: x[1])
feat_names = [x[0] for x in imp_sorted]
feat_vals = [x[1] for x in imp_sorted]
colors2 = ['red' if v < 0 else 'steelblue' for v in feat_vals]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(feat_names, feat_vals, color=colors2, edgecolor='black')
ax.set_xlabel('Mean reduction in C-index when permuted')
ax.set_title('Permutation-based predictor importance')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('figure2_permutation_importance.png', dpi=300)
plt.show()
files.download('figure2_permutation_importance.png')
